In [ ]:
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install seaborn
!pip install plotly
!pip install scipy

^C



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### **Libraries Import**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

### **Data Load**

In [ ]:
data = pd.read_csv("Dataset/financial_fraud_detection_&_risk_analytics.csv")

### **Data Structure**

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data.describe()

In [ ]:
data.info()

In [ ]:
data.shape

# **Data Cleaning & Preprocessing**

### **Data Quality**

In [ ]:
data.isnull().sum()
nullv = {}
for ds in data.columns:
    nullv[ds] = data[ds].isnull().sum()

nullv = pd.DataFrame({
    "Column": data.columns,
    "Null_Count": data.isnull().sum().values
})

nullv

In [ ]:
data.duplicated().sum()

### **Remove Duplicates**

In [ ]:
data.drop_duplicates(inplace=True)
data.shape

### **Split Column**

In [ ]:
data[["cinfo_f_name","cinfo_l_name"]] = data["Customer_Info"].str.split("|",expand=True)
data[["cprofile_f_name","cprofile_l_name","cprofile_email","cprofile_mono"]] = data["Customer_Profile"].str.split("~",expand=True)
data[["geo_City","geo_Country"]] = data["Geo_Info"].str.split(", ",expand=True)
data

### **Standardize Text Values**

In [ ]:
data["Payment_Method"] = data["Payment_Method"].str.lower().replace({
      "netbanking" : "NetBanking",
      "card" : "Card",
      "crypto" : "Crypto",
      "cash" : "Cash",
      "upi" : "UPI",
})

data["Country"] = data["Country"].str.lower().replace({
      "india" : "India",
      "usa" : "USA",
      "uk" : "UK",
      "uae" : "UAE",
      "singapore" : "Singapore",
      "canada" : "Canada",
      "australia" : "Australia",
      "germany" : "Germany",
})

data["City"] = data["City"].str.title().str.strip()
data["Merchant_Category"] = data["Merchant_Category"].str.title().str.strip()
data["Device_Type"] = data["Device_Type"].str.title().str.strip()
data["KYC_Status"] = data["KYC_Status"].str.title().str.strip()
data["Customer_Segment"] = data["Customer_Segment"].str.title().str.strip()
data["Email"] = data["Email"].str.lower().str.strip()
data["First_Name"] = data["First_Name"].str.title().str.strip()
data["Last_Name"] = data["Last_Name"].str.title().str.strip()
data["Full_Name"] = data["Full_Name"].str.title().str.strip()
data["Raw_Status"] = data["Raw_Status"].str.title().str.strip()

for clm in data.columns.to_list():
      if data[clm].dtype == "object":
            data[clm] = data[clm].str.strip()

### **Fix Data Type**

In [ ]:
data["Transaction_Date"] = pd.to_datetime(data["Transaction_Date"], format='mixed', dayfirst=True, errors='coerce')

data["Amount"] = pd.to_numeric(data["Amount"], errors='coerce')

data["Transaction_ID"] = data["Transaction_ID"].astype(object)
data["Customer_ID"] = data["Customer_ID"].astype(object)
data["Merchant_ID"] = data["Merchant_ID"].astype(object)
data["Payment_Method"] = data["Payment_Method"].astype(object)
data["Device_ID"] = data["Device_ID"].astype(object)
data["IP_Address"] = data["IP_Address"].astype(object)
data["Country"] = data["Country"].astype(object)
data["City"] = data["City"].astype(object)
data["Merchant_Category"] = data["Merchant_Category"].astype(object)
data["Device_Type"] = data["Device_Type"].astype(object)
data["KYC_Status"] = data["KYC_Status"].astype(object)
data["Customer_Segment"] = data["Customer_Segment"].astype(object)
data["Email"] = data["Email"].astype(object)
data["Phone_Number"] = data["Phone_Number"].astype(object)
data["Currency"] = data["Currency"].astype(object)
data["First_Name"] = data["First_Name"].astype(object)
data["Last_Name"] = data["Last_Name"].astype(object)
data["Full_Name"] = data["Full_Name"].astype(object)
data["Notes"] = data["Notes"].astype(object)
data["Raw_Status"] = data["Raw_Status"].astype(object)

data["Account_Age"] = pd.to_numeric(data["Account_Age"], errors='coerce').astype("Int64")
data["Transaction_Frequency"] = pd.to_numeric(data["Transaction_Frequency"], errors='coerce').astype("Int64")
data["Chargeback_Flag"] = pd.to_numeric(data["Chargeback_Flag"], errors='coerce').astype("Int64")
data["Fraud_Flag"] = pd.to_numeric(data["Fraud_Flag"], errors='coerce').astype("Int64")
data["Risk_Score"] = pd.to_numeric(data["Risk_Score"], errors='coerce').astype("Int64")
data["Login_Attempts"] = pd.to_numeric(data["Login_Attempts"], errors='coerce').astype("Int64")

In [ ]:
data.info()

### **Handle Missing Values**

In [ ]:
country_city_map = {
    "India": ["Mumbai", "Delhi", "Bangalore", "Ahmedabad", "Hyderabad", "Chennai", "Pune"],
    "USA": ["New York", "Los Angeles", "Chicago", "Houston", "San Francisco", "Miami", "Seattle", "Boston"],
    "UK": ["London", "Manchester", "Birmingham","Liverpool", "Leeds", "Bristol"],
    "UAE": ["Dubai", "Abu Dhabi", "Sharjah","Ajman", "Al Ain"],
    "Singapore": ["Singapore", "Tampines", "Jurong East","Woodlands", "Bedok"],
    "Canada": ["Toronto", "Vancouver", "Montreal","Calgary", "Ottawa", "Edmonton"],
    "Australia": ["Sydney", "Melbourne", "Brisbane","Perth", "Adelaide", "Canberra"],
    "Germany": ["Berlin", "Munich", "Hamburg","Frankfurt", "Cologne"]
}

merchant_segment_count = data.groupby("Merchant_Category")["Customer_Segment"].agg(lambda x: x.value_counts().index[0])
segment_merchant_count = data.groupby("Customer_Segment")["Merchant_Category"].agg(lambda x: x.value_counts().index[0])

merchant_segment_count, segment_merchant_count

In [ ]:
data["Device_ID"] = data["Device_ID"].fillna(
    data.groupby("IP_Address")["Device_ID"].transform("first")
)
data["IP_Address"] = data["IP_Address"].fillna(
    data.groupby("Device_ID")["IP_Address"].transform("first")
)

data["Merchant_Category"] = data["Merchant_Category"].fillna(data["Customer_Segment"])
data["Customer_Segment"] = data["Customer_Segment"].fillna(data["Merchant_Category"])

data.fillna({
      "Country" : data["geo_Country"],
      "City" : data["geo_City"],
      "Merchant_Location" : f"{data['City'], data['Country']}",
      "First_Name" : data["cinfo_f_name"],
      "Last_Name" : data["cprofile_l_name"],
      "Full_Name" : f"{data['First_Name']} {data['Full_Name']}",
}, inplace=True)

data["Currency"] = np.select(
    [
        data["Country"] == "India",
        data["Country"] == "USA",
        data["Country"] == "UK",
        data["Country"] == "UAE",
        data["Country"] == "Singapore",
        data["Country"] == "Canada",
        data["Country"] == "Australia",
        data["Country"] == "Germany"
    ],
    [
        "INR",
        "USD",
        "GBP",
        "AED",
        "SGD",
        "CAD",
        "AUD",
        "EUR"
    ],
    default="Unknown"
)

data["Device_Type"].fillna("Unknown", inplace=True)
data["Payment_Method"].fillna("Unknown", inplace=True)
data["Notes"].fillna("N/A")

### **Handle Outliers**

In [ ]:
data["Amount_Zscore"] = stats.zscore(data["Amount"].astype(float), nan_policy='omit')
amount_outlier = data[data["Amount_Zscore"].abs() > 3]

data["Risk_Score_Zscore"] = stats.zscore(data["Risk_Score"].astype(float), nan_policy='omit')
data["Transaction_Frequency_Zscore"] = stats.zscore(data["Transaction_Frequency"].astype(float), nan_policy='omit')

In [ ]:
data.loc[data["Account_Age"] < 0, "Account_Age"] = np.nan
data["Account_Age"] = data["Account_Age"].fillna(data["Account_Age"].median())

data["Transaction_Frequency"] = data["Transaction_Frequency"].fillna(
      data.groupby("Customer_Segment")["Transaction_Frequency"].transform("median")
)

data["Chargeback_Flag"] = data["Chargeback_Flag"].fillna(
      data.groupby("Chargeback_Flag")["Fraud_Flag"].agg(lambda x: x.mode().iloc[0])
)

data["Fraud_Flag"] = data["Fraud_Flag"].fillna(
      data.groupby("Fraud_Flag")["Risk_Score"].agg(lambda x: x.mode().iloc[0])
)

data["Risk_Score"] = data["Risk_Score"].fillna(data[["Fraud_Flag","Chargeback_Flag"]].median())

data["Login_Attempts"] = data["Login_Attempts"].fillna(
      data.groupby("Login_Attempts")["Fraud_Flag"].transform("median")
)

### **Drop Unvalid Columns**

In [ ]:
data.head(3)

##### **Drop Remains Null Values Rows From Dataset**

In [ ]:
data.drop(["cinfo_f_name","cinfo_l_name","cprofile_f_name","cprofile_l_name","cprofile_email","cprofile_mono","geo_City","geo_Country"], axis=1, inplace=True)
data.drop(["Customer_Info", "Merchant_Location", "Customer_Profile","Geo_Info"], axis=1, inplace=True)
# data.drop(amount_outlier, index=1)

##### **Drop Unused Column For Data Visualization**

In [ ]:
data.drop(["Email","Phone_Number"], axis=1, inplace=True)

### **Export Clean Dataset**

In [ ]:
data.info()

In [ ]:
data.to_csv("Dataset/cleaned.csv", index=False)

# **Data Visualization**

#### **1. Fraud vs Non-Fraud Transactions**

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=data,x="Fraud_Flag")
plt.xlabel("Fraud Status")
plt.ylabel("Frequency")
plt.title("Fraud vs Non-Fraud Transactions")
plt.show()

#### **2. Transaction Amount Distribution**

In [ ]:
sns.histplot(data["Amount"], bins=5)
plt.title("Transaction Amount Distribution")
plt.show()

#### **3. Risk Score Distribution**

In [ ]:
plt.figure(figsize=(8,6))
sns.histplot(data["Risk_Score"], bins=20)
plt.title("Risk Score Distribution")
plt.show()

#### **4. Fraud by Payment Method**

In [ ]:
paymentmethod_dist = data["Payment_Method"].value_counts().reset_index()
paymentmethod_dist.columns = ["Payment Method","Fraud Ratio"]
fig = px.pie(paymentmethod_dist, names="Payment Method", values="Fraud Ratio", title="Fraud Ratio by Payment Method")
fig.show()

#### **5. Fraud by Country**

In [ ]:
fraud_country = (
      data.groupby("Country")["Fraud_Flag"]
      .sum()
      .reset_index()
)
fig = px.bar(fraud_country, x="Country", y="Fraud_Flag", title="Fraud Transactions by Country")
fig.show()

#### **6. Fraud by Merchant Category**

In [ ]:
fraud_cat = (
      data.groupby("Merchant_Category")["Fraud_Flag"]
      .sum()
      .reset_index()
)
fig = px.treemap(fraud_cat, path=["Merchant_Category"], values="Fraud_Flag", title="Fraud by Merchant Category")
fig.show()

#### **7. Fraud Trend Over Time**

In [ ]:
plt.figure(figsize=(10,6))
daily_fraud = (
      data.groupby("Transaction_Date")["Fraud_Flag"]
      .sum()
)
daily_fraud.plot()
plt.title("Daily Fraud Trend")
plt.show()

#### **8. Risk Score vs Amount**

In [ ]:
per30data = int(data.shape[0] * 0.15)
fig = px.scatter(data.sample(per30data), x="Amount", y="Risk_Score", color="Fraud_Flag", title="Risk Score vs Amount")
fig.show()

#### **9. Amount Boxplot**

In [ ]:
sns.boxplot(x=data["Amount"])
plt.title("Amount Outliers")
plt.show()

#### **10. Risk Score Boxplot by Fraud**

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=data,
    x="Fraud_Flag",
    y="Risk_Score"
)

plt.title("Risk Score Distribution by Fraud Status")
plt.xlabel("Fraud Status")
plt.ylabel("Risk Score")

plt.xticks(
    [0, 1],
    ["Non-Fraud", "Fraud"]
)

plt.show()

#### **11. Correlation Heatmap**

In [ ]:
numeric_cols = data.select_dtypes(include=["int64","float64"])
plt.figure(figsize=(10,8))
sns.heatmap(numeric_cols.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

#### **12. Fraud by Device Type**

In [ ]:
device = (
      data.groupby("Device_Type")["Fraud_Flag"]
      .sum()
      .reset_index()
)
fig = px.pie(device, names="Device_Type", values="Fraud_Flag", title="Fraud by Device Type")
fig.show()

#### **13. Top Suspicious Devices**

In [ ]:
n = 10

plt.figure(figsize=(8,6))
suspicious_devices = (
    data.groupby("Device_ID")["Fraud_Flag"]
    .sum()
    .sort_values(ascending=False)
    .head(n)
)

suspicious_devices.plot(kind="bar")
plt.title("Top Fraud Devices")
plt.show()

#### **14. KYC Status Distribution**

In [ ]:
sns.histplot(data=data, x="KYC_Status")
plt.title("KYC_Status")
plt.show()

#### **15. Fraud by KYC Status**

In [ ]:
kyc = (
      data.groupby("KYC_Status")["Fraud_Flag"]
      .mean()*100
)
kyc.plot(kind="bar")
plt.ylabel("Fraud Rate %")
plt.title("Fraud Rate by KYC Status")
plt.show()

#### **16. Customer Segment Analysis**

In [ ]:
segment = (
      data.groupby("Customer_Segment")["Fraud_Flag"]
      .mean()*100
)
segment.plot(kind="bar")
plt.ylabel("Fraud Rate %")
plt.title("Fraud Rate by Customer Segment")
plt.show()

#### **17. Transaction Frequency vs Risk Score**

In [ ]:
plt.figure(figsize=(12,12))
plt.title("Transaction Frequency v/s Risk Score")
sns.scatterplot(
    data=data.sample(10000),
    x="Transaction_Frequency",
    y="Risk_Score",
    hue="Fraud_Flag"
)
plt.show()

#### **18. Missing Values Heatmap**

In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(data.isnull(), cbar=False)
plt.title("Missing Values Heatmap")
plt.show()

#### **19. Outlier Detection**

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=data, y="Amount", ax=ax[0])
ax[0].set_title("Amount Outliers")

sns.boxplot(data=data, y="Account_Age", ax=ax[1])
ax[1].set_title("Account Age Outliers")

sns.boxplot(data=data, y="Transaction_Frequency", ax=ax[2])
ax[2].set_title("Transaction Frequency Outliers")

plt.tight_layout()
plt.show()

#### **20. Fraud Rate by Risk Category**

In [ ]:
data["Risk_Category"] = pd.cut(
    data["Risk_Score"],
    bins=[0, 20, 40, 60, 80, 100],
    labels=[
        "Very Low",
        "Low",
        "Medium",
        "High",
        "Critical"
    ]
)

fraud_rate = (
    data.groupby("Risk_Category")["Fraud_Flag"]
    .mean()
    .reset_index()
)

fraud_rate["Fraud_Rate"] = fraud_rate["Fraud_Flag"] * 100

fig = px.bar(
    fraud_rate,
    x="Risk_Category",
    y="Fraud_Rate",
    color="Fraud_Rate",
    category_orders={
        "Risk_Category": [
            "Very Low",
            "Low",
            "Medium",
            "High",
            "Critical"
        ]
    },
    text_auto=".2f",
    title="Fraud Rate by Risk Category (%)"
)

fig.show()

#### **21. Pair Plot**

In [ ]:
cols = [
    "Amount",
    "Risk_Score",
    "Transaction_Frequency",
    "Login_Attempts",
    "Fraud_Flag"
]

sns.pairplot(
    data[cols],
    diag_kind="kde"
)
plt.show()

In [ ]:
sample_size = int(data.shape[0] * 0.15)
sample_df = data.sample(sample_size, random_state=42)
sns.pairplot(
    sample_df[cols],
    hue="Fraud_Flag",
    diag_kind="hist"
)

plt.show()